<a href="https://colab.research.google.com/github/RoihansLab/Data-Mining-Project/blob/main/youtube_comment_sentiment/Classification/finetune_transformer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Percobaan: Fine-tuning Transformer di Google Colab**

Notebook ini **terpisah** dari notebook utama (`classification_modeling.ipynb`). Tujuannya hanya membandingkan: apakah *fine-tuning* model Transformer dengan GPU Colab bisa mengalahkan model akhir (**e5-large + TF-IDF + Logistic Regression**, macro-F1 sekitar 0,66)?

**Supaya perbandingannya adil:**
- Memakai data dan label yang sama (`youtube_comments_label_final.csv`).
- Memakai **data uji yang sama persis** (3.927 komentar, dibagi 80:20, `random_state=42`, *stratified*).
- Data uji hanya dipakai **sekali di akhir**. Pemilihan epoch terbaik memakai **10% data latih** sebagai data validasi.

**Cara pakai:**
1. Pilih *Runtime → Change runtime type → T4 GPU*.
2. Upload `youtube_comments_label_final.csv` lewat ikon folder di sidebar kiri.
3. Jalankan semua sel (*Runtime → Run all*). Dengan `xlm-roberta-large` sekitar 20-30 menit.

## **1. Import Library**

In [ ]:
!pip install -q Sastrawi transformers accelerate

import os, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (pilih T4 GPU di Runtime)")

## **2. Data dan Pembagian yang Sama dengan Notebook Utama**

Di notebook utama, 8 komentar dibuang karena menjadi kosong setelah *removing* dan *stopword* (Bab 6.3-6.4). Sel ini mengulang aturan yang sama **hanya untuk menentukan baris mana yang dibuang**, supaya pembagian data latih dan uji sama persis. Transformer sendiri membaca **teks komentar asli** (kolom `comment`), karena model ini memahami kalimat lengkap.

In [ ]:
for path in ["/content/youtube_comments_label_final.csv", "youtube_comments_label_final.csv", "../../datasets/youtube_comments_label_final.csv"]:
    if os.path.exists(path):
        break
df = pd.read_csv(path)

# Aturan Bab 6.2-6.4 notebook utama (hanya untuk menyamakan baris yang dibuang)
teks = df["comment"].str.lower()
teks = teks.str.replace(r"https?://\S+|www\.\S+", " ", regex=True).str.replace(r"[^a-z\s]", " ", regex=True)
negasi = {"tidak", "belum", "nggak", "not", "no", "nor", "never", "nothing", "none", "cannot"}
stopword = (set(StopWordRemoverFactory().get_stop_words()) | set(ENGLISH_STOP_WORDS)) - negasi
teks = teks.apply(lambda t: " ".join(kata for kata in t.split() if kata not in stopword))
df = df[teks != ""].reset_index(drop=True)

# Pembagian sama dengan notebook utama, lalu 10% data latih dipakai sebagai validasi
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
df_latih, df_val = train_test_split(df_train, test_size=0.1, random_state=42, stratify=df_train["label"])
print("Total:", len(df), "| latih:", len(df_latih), "| validasi:", len(df_val), "| uji:", len(df_test))

## **3. Tokenisasi**

`MODEL` bisa diganti. Pilihan yang cocok untuk komentar bahasa Indonesia + Inggris:

| Model | Ukuran | Catatan |
|---|---|---|
| `xlm-roberta-large` | 560 juta parameter | paling kuat, perlu GPU T4 (sekitar 20-30 menit) |
| `xlm-roberta-base` | 280 juta parameter | lebih cepat (sekitar 8-10 menit) |
| `microsoft/mdeberta-v3-base` | 280 juta parameter | alternatif, kadang tidak stabil |

Komentar dipotong maksimal **128 token** (cukup untuk sebagian besar komentar).

In [ ]:
MODEL = "xlm-roberta-large"
label_ke_id = {"negatif": 0, "perlu ditinjau": 1, "positif": 2}
id_ke_label = {v: k for k, v in label_ke_id.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL)


class DataKomentar(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tokenizer(d["comment"].tolist(), truncation=True, max_length=128, padding="max_length")
        self.y = d["label"].map(label_ke_id).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i])
        return item


data_latih, data_val, data_uji = DataKomentar(df_latih), DataKomentar(df_val), DataKomentar(df_test)

## **4. Fine-tuning**

- **Bobot label** (`balanced`): label yang sedikit (positif dan negatif) diberi bobot lebih besar di *loss*, sama seperti `class_weight="balanced"` di Logistic Regression.
- Dilatih **5 epoch**. Setiap akhir epoch dinilai di data validasi, lalu **epoch dengan macro-F1 validasi tertinggi** yang dipakai.
- `fp16=True` supaya hemat memori dan lebih cepat di GPU T4.

In [ ]:
bobot = torch.tensor(compute_class_weight("balanced", classes=np.arange(3), y=data_latih.y), dtype=torch.float)


class TrainerBerbobot(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        label = inputs.pop("labels")
        keluaran = model(**inputs)
        loss = torch.nn.functional.cross_entropy(keluaran.logits, label, weight=bobot.to(keluaran.logits.device))
        return (loss, keluaran) if return_outputs else loss


def hitung_metrik(hasil):
    pred = hasil.predictions.argmax(axis=1)
    return {"macro_f1": f1_score(hasil.label_ids, pred, average="macro"), "akurasi": accuracy_score(hasil.label_ids, pred)}


model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3, id2label=id_ke_label, label2id=label_ke_id)
pengaturan = TrainingArguments(
    output_dir="hasil_finetune", learning_rate=1e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=5, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, fp16=torch.cuda.is_available(), seed=42, report_to="none")
trainer = TrainerBerbobot(model=model, args=pengaturan, train_dataset=data_latih, eval_dataset=data_val, compute_metrics=hitung_metrik)
trainer.train()

## **5. Evaluasi di Data Uji (sekali di akhir)**

Bandingkan dengan model akhir di notebook utama: **e5-large + TF-IDF + Logistic Regression, macro-F1 sekitar 0,66 dan akurasi sekitar 0,72**.

In [ ]:
hasil = trainer.predict(data_uji)
y_uji = np.array([id_ke_label[i] for i in hasil.label_ids])
pred_uji = np.array([id_ke_label[i] for i in hasil.predictions.argmax(axis=1)])

print("Model:", MODEL)
print("Macro-F1:", round(f1_score(y_uji, pred_uji, average="macro"), 3), "| Akurasi:", round(accuracy_score(y_uji, pred_uji), 3))
print(classification_report(y_uji, pred_uji, digits=3))
ConfusionMatrixDisplay.from_predictions(y_uji, pred_uji, cmap="Blues", xticks_rotation=20)
plt.title(f"Confusion Matrix {MODEL}")
plt.show()

**Cara membaca hasil:**
- Jika macro-F1 **lebih tinggi dari sekitar 0,66**, fine-tuning Transformer lebih baik untuk data ini.
- Jika **setara atau lebih rendah**, berarti batas utamanya ada pada jumlah data dan kualitas label, bukan pada kekuatan model.
- Selisih kecil (kurang dari sekitar 0,03) belum tentu berarti, karena masih di dalam variasi acak data uji (rentang bootstrap model akhir sekitar 0,62-0,69).
- Hasil fine-tuning bisa sedikit berbeda setiap dijalankan, walaupun *seed* sudah dikunci 42, karena perhitungan GPU tidak selalu deterministik.